# Lager

Liest den Bestand, prüft ihn und schreibt die Fehlmengen. Jede Code-Zelle wird im Diagramm
eine Funktion `<cell n>`.

In [1]:
%pip install pandas
import pandas as pd
import sqlite3

MINDEST = 5
db = sqlite3.connect("lager.db")

In [2]:
%%sql
CREATE TABLE IF NOT EXISTS artikel (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    bestand INTEGER NOT NULL
);
CREATE TABLE IF NOT EXISTS fehlmenge (
    artikel INTEGER NOT NULL REFERENCES artikel(id),
    menge INTEGER NOT NULL
);

## Prüfen

In [3]:
def fehlt(bestand):
    if bestand < 0:
        raise ValueError("Bestand negativ")
    return max(MINDEST - bestand, 0)

zeilen = db.execute("SELECT id, bestand FROM artikel").fetchall()
knapp = 0
for id_, bestand in zeilen:
    menge = fehlt(bestand)
    if menge > 0:
        db.execute("INSERT INTO fehlmenge (artikel, menge) VALUES (?, ?)", (id_, menge))
        knapp += 1
print(f"{knapp} Artikel unter dem Mindestbestand")

3 Artikel unter dem Mindestbestand


In [4]:
!ls -l lager.db
db.commit()
db.close()